## See "Hard Mining Negatives for Semantic Similarity"

https://www.kaggle.com/code/jithinanievarghese/hard-mining-negatives-for-semantic-similarity#Load-Data-and-preprocess-data

In [11]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/playground')
)
#only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import utils as ut
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import numpy as np 
import pandas as pd 
import csv

In [5]:
def preprocess_text(text):
    """
    clean white space and lower case the text
    """
    return " ".join(text.split()).lower()

In [12]:
df = pd.read_json('../data/trn.json')

df.drop_duplicates(subset=['anchor', 'positive'], inplace=True)
# df.drop_duplicates(subset=['anchor'], inplace=True)
# df.drop_duplicates(subset=['positive'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.anchor = df['anchor'].apply(lambda x: preprocess_text(x))
df.positive = df['positive'].apply(lambda x: preprocess_text(x))

# df.head()

# Embed the columns of interest

### Get list of positives and anchors

In [7]:
%%time
from sentence_transformers import InputExample
from tqdm.auto import tqdm  # so we see progress bar
def getsentencelists(df,cols):
    '''
    param: df dataframe
    param: cols list of columns in dataframe to return lists from
    return: tuple of lists, each list is a string of all strings in column
     of InputExample objects
     ex.
     cols=['positive','anchor']
    positives, anchors = getsentencelists(df,cols)
    ''' 
    res={}  
    for col in cols:
        res[col]=[]
    for _,row in tqdm(df.iterrows()):
        for col in cols:
            res[col].append(row[col])
    return (res[col] for col in cols)

cols=['positive','anchor']
positives, anchors = getsentencelists(df,cols)


35258it [00:00, 36377.09it/s]

CPU times: user 964 ms, sys: 2.92 ms, total: 967 ms
Wall time: 972 ms


## the positive column has a lot of repeats in it

In [8]:
df['positive'].nunique()

3346

### generate embeddings

In [9]:
%%time

import logging
import torch
import numpy as np

from sentence_transformers import LoggingHandler, SentenceTransformer

#### Just some code to print debug information to stdout
np.set_printoptions(threshold=100)

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()]
)
#### /print debug information to stdout


# Load pre-trained Sentence Transformer Model. It will be downloaded automatically
model = SentenceTransformer("all-MiniLM-L6-v2",device="cuda:0" if torch.cuda.is_available() else "cpu",)

# Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
positive_embeddings = model.encode(positives, convert_to_tensor=True)
anchor_embeddings = model.encode(anchors, convert_to_tensor=True)


2024-07-09 17:40:09 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 1102/1102 [00:09<00:00, 118.15it/s]

CPU times: user 25min 13s, sys: 42.3 s, total: 25min 55s
Wall time: 24.6 s


### get similarity score matrix

In [3]:
scores2=model.similarity(anchor_embeddings, positive_embeddings)

NameError: name 'model' is not defined

In [2]:
a=(scores2-scores).cpu().detach().numpy()

NameError: name 'scores2' is not defined

In [10]:
# We use cosine-similarity 
scores=model.similarity(anchor_embeddings, positive_embeddings)
# print(scores.device)

### Hard negative mine the positives for similar positives

This assummes the model has been fine tuned on the dataset first

In [51]:
#the following runs on GPU
import torch
from itertools import compress
from tqdm.auto import tqdm
from numba import njit
import numpy as np

def get_scores_processed(scores, high=0.65):
    """
    Process the scores and generate a mask based on a threshold.

    Parameters:
    scores (torch.Tensor): The input scores.
    high (float, optional): The threshold value. Defaults to 0.65.

    Returns:
    tuple: A tuple containing two numpy arrays - the processed scores and the mask.

    """
    
    # Get the entries below high
    scores_mask = (scores < high).to(scores.device)

    # Set diagonal to False (don't want true positive included in the hard negatives)
    mask = (torch.eye(scores_mask.shape[0], scores_mask.shape[0]) > 0).to(scores.device)
    scores_mask.masked_fill_(mask, False)
   
    return (scores.cpu().detach().numpy(), scores_mask.cpu().detach().numpy())

#the following is compiled into c and runs on a cpu
@njit
def get_results(candidates, scores, true_positive, low=0.5, topn=20):
    """
    Returns a list of results based on the given candidates, scores, true positive, and optional parameters.

    Parameters:
    candidates (list): A list of candidate positions.
    scores (list): A list of cosign similarity scores corresponding to the candidates.
    true_positive (str): The true positive string.
    low (float, optional): The threshold score value. Defaults to 0.5.
    topn (int, optional): The maximum number of results to return. Defaults to 20.

    Returns:
    list: A list of results that meet the criteria.

    """
    res = [pos for score, pos in candidates[:topn] if score >= low]

    # Remove all duplicate entries
    res = list(set(res))

    # Remove all hits that are the same as the true positive
    res = [val for val in res if val != true_positive]    
    return res

@njit
def get_hard_negatives_CPU(scores,scores_mask,positives,low=0.5, high=0.65, topn=20):
    """
    Train a sentencetransformer model, get its average similarity score, use range around that average for hard
    negatives
    expects scores to be nxn matrix of similarity scores, nparray
    expects positives to be a list of n strings
    expects high to be floats denoting the max acceptable similarity score
    topn: int, number of hard negatives to return from torch.top_k
    Get pairs of indices with low<= score <= high
    returns: list of list of positives whose similarity score is between low and high
    """
    
      # use scores_mask to select hard negatives
    hard_negatives=[]
    hn_found=0
    poor_hn_found=0

    for i,row in enumerate(scores_mask):
        #get all the positives and their scores
        # #make sure none of these positives are the same as the true positive
        # #this happens when you derive multiple queries from the same positive
        canditates=[(scores[i,j],positives[j]) for j in range(len(row)) if row[j]==True]

        #sort it by score
        true_positive=positives[i]
        canditates.sort(key=lambda x: x[0],reverse=True)
        # print(f'tup={tup}')

        res=get_results(canditates, scores, true_positive, low,topn)

        if(len(res)==0):
            #nothing found between high and low, take results from 0 ->low
            res=get_results(canditates, scores, true_positive, low=0,topn=topn)
            poor_hn_found+=1
        else:
            hn_found+=1
        
        hard_negatives.append(res)
    print(f"hn_found={hn_found}, poor_hn_found={poor_hn_found}")
    return hard_negatives


In [46]:
# small test case for above
ns=500
high=0.65
scores2=scores[:ns,:ns]
ghn3=get_hard_negatives_CPU(*(get_scores_processed(scores2,high)),positives[:ns],high=high)

hn_found=499, poor_hn_found=1


In [52]:
high=0.65
ghn3=get_hard_negatives_CPU(*(get_scores_processed(scores,high)),positives,high=high)

hn_found=29636, poor_hn_found=5622


In [49]:
ghn3[:5]

[['3. confidentiality the parties hereto agree that each shall treat confidentially the terms and conditions of this agreement and all information provided by each party to the other regarding its business and operations. all confidential information provided by a party hereto, including nonpublic personal information (regulated pursuant to regulation s-p), shall be used by any other party hereto solely for the purpose of rendering services pursuant to this agreement and, except as may be required in carrying out this agreement, shall not be disclosed to any third party, without the prior consent of such providing party. the foregoing shall not be applicable to any information that is publicly available when provided or thereafter becomes publicly available other than through a breach of this agreement, or that is required to be disclosed by any regulatory authority, any authority or legal counsel of the parties hereto, by judicial or administrative process or otherwise by applicable l

In [53]:
#lets see how they look
# for p in ghn[:20]: print(len(p))
print(f'ANCHOR={anchors[0]}')
print(f'POSITIVE={positives[0]}')
print(f'HARDNEG={ghn3[0]}')

ANCHOR=what safeguards are in place to protect the information obtained from third-party sources?
POSITIVE=information we collect from other sources we may also receive information from other sources and combine that with information we collect through our services. for example: if you choose to link, create, or log in to your uber account with a payment provider (e.g., google wallet) or social media service (e.g., facebook), or if you engage with a separate app or website that uses our api (or whose api we use), we may receive information about you or your connections from that site or app.
HARDNEG=['3. confidentiality the parties hereto agree that each shall treat confidentially the terms and conditions of this agreement and all information provided by each party to the other regarding its business and operations. all confidential information provided by a party hereto, including nonpublic personal information (regulated pursuant to regulation s-p), shall be used by any other party h

In [ ]:
ghn[:6]

[['2.5 neither party shall be required to keep confidential any information which is, or becomes, publicly available, is independently developed by either party outside the scope of this agreement, or is rightfully obtained from third parties.'],
 ["3.3 warranty. company shall at all times make reasonable efforts to maintain quality control and to deliver products to distributor which, when received by distributor, or, as the case may be, the end-user, are properly and adequately packaged and contained, fully assembled (except for miscellaneous components which may be shipped separately to prevent damage in transit), fully functional and otherwise in conformance with the warranties set forth herein. company warrants that the products will be designed, manufactured, constructed, assembled and packaged in a workmanlike manner and that such products shall be fully functional and fit for their intended purposes. company further warrants that the products sold hereunder shall be free from d

### add to dataframe

In [ ]:
#drop all hard negatives except for the first one
df.drop(columns=['hard_negative'],inplace=True)
# df.drop(columns=['most_dissimilar_context','id'],inplace=True)

In [ ]:
df.head()

,anchor,positive,hard_negative
0,what safeguards are in place to protect the in...,information we collect from other sources we m...,[2.5 neither party shall be required to keep c...
1,is there a guarantee from the manufacturers re...,each of the suppliers warrants that the produc...,[3.3 warranty. company shall at all times make...
2,what type of authorization has the video confe...,skype hereby grants to online bvi and the comp...,[abbvie may use one (1) or more of its affilia...
3,can the blockchain administrator arrange for t...,(a) the fund hereby employs the blockchain adm...,[(g) the blockchain administrator is hereby au...
4,what happens if a party fails to retain record...,each party will retain such records for at lea...,[each party will retain such records for at le...


In [ ]:

df['hard_negative']=ghn

In [ ]:
#how many unique positives do we have?
df.positive.nunique()

3346

In [ ]:
j=0
for i in range(len(df)):
    if(df.iloc[i,1]==df.iloc[i,2][0]):
        j+=1
print(j)


IndexError: list index out of range

In [ ]:
print(df.iloc[4,1])
print(df.iloc[4,2][0])

each party will retain such records for at least three (3) years following expiration or termination of this agreement or such longer period as may be required by applicable law or regulation.
each party will retain such records for at least three (3) years following expiration or termination of this agreement or such longer period as may be required by applicable law or regulation.


In [ ]:
#to convert dataset column type
import datasets as ds
def add_list_to_dataset(df,colname:str, data:list):
    '''
    adds column, colname to a dataframe
    colname column name to add
    data, lists of lists. Each list has hard negative samples
    
    ex:

    '''
   
    #add back as list
    df = df.add_column(colname, data)

    return df


# Junk

In [ ]:
# def get_hard_negatives(scores,positives,low=0.5, high=0.65, topn=20):
#     """
#     Train a sentencetransformer model, get its average similarity score, use range around that average for hard
#     negatives
#     expects scores to be nxn matrix of similarity scores
#     expects positives to be a list of n strings
#     expects high to be floats denoting the max acceptable similarity score
#     topn: int, number of hard negatives to return from torch.top_k
#     Get pairs of indices with low<= score <= high
#     returns: list of list of positives whose similarity score is between low and high
#     """
#     high=torch.tensor([high])
    
#     #put the following on the same device as scores
#     if scores.is_cuda:
#         high=high.to(scores.device)

#     # Get the entries between low and high
#     h=torch.lt(scores,high)
#     scores_mask=(h).tolist()
#     print(f'len(scores_mask)={len(scores_mask)}, scores_mask[0][:5]={scores_mask[0][:5]}')

#     # use scores_mask to select hard negatives
#     hard_negatives=[]
#     for i,row in tqdm(enumerate(scores_mask)):
#         #row[i] contains the true positive, therefore do not use it
#         row[i]=False

#         #get all the positives and their scores
#         # tup=[(scores[i,j].item(),positives[j]) for j in range(len(row)) if row[j]==True]

#         # #make sure none of these positives are the same as the true positive
#         # #this happens when you derive multiple queries from the same positive
#         # tup=[(score,pos) for score,pos in tup if pos!=positives[i]]

#         # #sort it by score
#         # tup.sort(key=lambda x: x[0],reverse=True)
#         # # print(f'tup={tup}')

#         # #get all acceptable hard negatives
#         # res=[pos for score,pos in tup[:topn] if score>=low]
                
#         # if(len(res)==0):
#         #     print(f"no HN for row={i}, length of ")
        
#         # hard_negatives.append(res)
#     return hard_negatives

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Cannot infer the type of variable 'hard_negatives', have imprecise type: list(undefined)<iv=None>. 

For Numba to be able to compile a list, the list must have a known and
precise type that can be inferred from the other variables. Whilst sometimes
the type of empty lists can be inferred, this is not always the case, see this
documentation for help:

https://numba.readthedocs.io/en/stable/user/troubleshoot.html#my-code-has-an-untyped-list-problem


File "../../../../tmp/ipykernel_2352648/4225380099.py", line 31:
<source missing, REPL/exec in use?>


In [ ]:

# import torch
# from itertools import compress
# from tqdm.auto import tqdm
# from numba import njit
# import numpy as np

# @njit
# def get_hard_negatives(scores,positives,low=0.5, high=0.65, topn=20):
#     """
#     Train a sentencetransformer model, get its average similarity score, use range around that average for hard
#     negatives
#     expects scores to be nxn matrix of similarity scores
#     expects positives to be a list of n strings
#     expects high to be floats denoting the max acceptable similarity score
#     topn: int, number of hard negatives to return from torch.top_k
#     Get pairs of indices with low<= score <= high
#     returns: list of list of positives whose similarity score is between low and high
#     """
#     # high=torch.tensor([high])
    
#     #put the following on the same device as scores
#     # if scores.is_cuda:
#     #     high=high.to(scores.device)

#     # Get the entries between low and high
#     scores_mask=scores<high
#     # scores_mask=(h).tolist()
#     print(f'len(scores_mask)={len(scores_mask)}')

#     # use scores_mask to select hard negatives
#     hard_negatives=[]
#     hn_found=0
#     poor_hn_found=0
#     for i,row in enumerate(scores_mask):
#    #the following is compiled into c and runs on a cpu     #row[i] contains the true positive, therefore do not use it
#         row[i]=False
#         true_positive=positives[i]

#         #get all the positives and their scores
#         # #make sure none of these positives are the same as the true positive
#         # #this happens when you derive multiple queries from the same positive
#         canditates=[(scores[i,j],positives[j]) for j in range(len(row)) if row[j]==True and positives[j]!=true_positive]

#         #sort it by score
#         canditates.sort(key=lambda x: x[0],reverse=True)
#         # print(f'tup={tup}')

#         # #get all acceptable hard negatives
#         res=[pos for score,pos in canditates[:topn] if score>=low]

#         #remove all duplicate entries
#         res=list(set(res))
                
#         if(len(res)==0):
#             # print(f"no hard negative for row={i}, using top 1 instead with score={str(canditates[0][0])}")
#             res=[pos for score,pos in canditates[:1]]
#             poor_hn_found+=1
#         else:
#             hn_found+=1
        
#         hard_negatives.append(res)
#     print(f"hn_found={hn_found}, poor_hn_found={poor_hn_found}")
#     return hard_negatives


In [ ]:

# test case for above
# ns=5
# scores2=model.similarity(anchor_embeddings[:ns], positive_embeddings[:ns])
# print(scores2)
# high=torch.tensor([0.3]).to(scores2.device)
    
# # set everything thats too high to 0
# scores2_mask=(scores2<high)
# print(scores2_mask)
# # 

# #set diagonal to False
# mask = (torch.eye(ns, ns)>0).to(scores2.device)
# scores2_mask.masked_fill_(mask, 0)
# print(scores2_mask)
# scores2_mask=scores2_mask.float()
# print(scores2_mask)

# l=['a 1','b 2','c 3','d 4','e 5']
# l_indexes=[i for i in range(len(l))]
# print(l_indexes)
# # positives=torch.tensor(['a 1','b 2','c 3','d 4','e 5']).to(scores2.device)
# positives=torch.tensor([1,2,3,4,5]).to(scores2.device)
# scores2_mask*l_indexes

# scores2*scores2_mask


In [7]:
# class HardMineNegatives():
#     """
#     Hard-mining Negatives for training a semantic similairty task with Triplet Loss.
#     Here we find the nearest negatives of a query in a search pool 
#     by using sentence transformer model embeddings and cosine similarity ratio.
#     param: model_path: path of sentence transformer model
#     param: search_max_threshold:  maximimum cosine similarity ratio
#     param: search_min_threshold: minimum cosine similarity ratio
#     param: search_limit: total length of data in which we want to search, only if search pool length is very high
#     param: top_n_results: number of top nearest negative  to be returned, default is 1
#     """
#     def __init__(self, model_path: str, **kwargs):
#         self.model = SentenceTransformer(model_path)
#         self.search_max_threshold = kwargs['search_max_threshold'],
#         self.search_min_threshold = kwargs['search_min_threshold']
#         self.search_limit = kwargs.get('search_limit')
#         self.top_n_results = kwargs.get('top_n_results') if kwargs.get('top_n_results') else 1

#     def get_hard_mined_negatives(self, anchor: str,  search_pool:np.ndarray):
#         """
#         to retrieve embeddings from sentence transformer model for anchor and sentences in search pool,
#         find the cosine similairty ratio between the  anchor and search pool sentences,
#         apply search thresholds and return the top nearest negatives based on the highest
#         cosine similarity scores.
#         if no data is found in between the self.search_max_threshold and self.search_min_threshold ratios,
#         we will take the results between 0 and less than self.search_min_threshold ratios (this is an extreme case)

#         param: anchor: source text to which we need to find the nearest negative
#         param: search_pool: numpy array of sentences from which
#                we need to find the cosine similarity ratios with the anchor.
#                any meta value for sentences can be given after next index of
#                sentence, in the form
#                search_pool = array([
#                     ['apple iphone 8 256 gb gold', "mobile", "1001"],
#                     ['apple iphone 7 plus 32gb silver', "1002"]])
#                where "mobile", "1001" are meta values,
#                the returned results will contain the respective cosine similarity
#                ratio at the last index of each sentence array
#                result = array([
#                     ['apple iphone 8 256 gb gold', "mobile", "1001", 69.5],
#                     ['apple iphone 7 plus 32gb silver', "1002", 70.5]])
#                where 69.5 and 70.5 are cosine similarity ratios.
#         """
#         self.search_limit = self.search_limit if self.search_limit else search_pool.shape[0]
#         search_pool = search_pool[: self.search_limit]
#         # shuffle data to search in random pool of data, in case of search limit less than the total length
#         np.random.shuffle(search_pool)
#         sentences = [anchor] + [row[0] for row in search_pool]
#         embeddings = self.model.encode(sentences, convert_to_tensor=False)
#         source_vector = embeddings[0]
#         # calculate the cosine similairty with the other sentences in search pool
#         similarity = [round(util.cos_sim(source_vector, embed).numpy()[0][0]*100, 2) for embed in embeddings[1:]]
#         similarity = np.array(similarity)
#         negative_indices = np.where((similarity <= self.search_max_threshold) & (similarity >= self.search_min_threshold))
#         if not negative_indices[0].shape[0]:
#             negative_indices = np.where((similarity < self.search_min_threshold) & (similarity >= 0))
#         negative_indices = negative_indices[0]
#         # take respective selected indices
#         search_pool = np.take(search_pool, negative_indices, axis=0)
#         similarity = np.take(similarity, negative_indices, axis=0)
#         # reshape to concatenate with meta values of search pool
#         similarity = similarity.reshape(-1, 1)
#         # concat the ratio to the meta values of search pool
#         search_pool = np.concatenate((search_pool, similarity), axis=1)
#         # sort the data in descending order
#         search_pool = search_pool[search_pool[:, -1].argsort()][::-1]
# #         return search_pool[:self.top_n_results]
# model_path = 'sentence-transformers/all-MiniLM-L6-v2'
# obj = HardMineNegatives(
#     model_path=model_path,
#     search_max_threshold=65,
#     search_min_threshold=50,
#     search_limit=None,
#     top_n_results=1)
# final_results = []
# for row in tqdm(df.to_dict('records')):
#     anchor = row['anchor']
#     search_pool = df[df.anchor != anchor]
#     search_pool.reset_index(drop=True, inplace=True)
#     search_pool = search_pool.drop_duplicates()
#     search_pool = search_pool.loc[:, ['positive']]
#     search_pool = search_pool.to_numpy()
#     print(f'Mining negatives for "{anchor}"')
#     top_results = obj.get_hard_mined_negatives(anchor, search_pool)
#     anchor = np.array([anchor]).reshape(-1, 1)
#     anchor = np.repeat(anchor, repeats=len(top_results), axis=0)
#     results_with_meta = np.concatenate((anchor, top_results), axis=1)
#     final_results.extend(results_with_meta.tolist())

In [ ]:
# import torch

# a = torch.Tensor([[12, 1, 0, 0],
#                   [4, 9, 21, 1],
#                   [10, 2, 1, 0]])

# b = torch.rand(3, 4, 8)

# print('a_size', a.size())
# # a_size torch.Size([3, 4])
# print('b_size', b.size())
# # b_size torch.Size([3, 4, 8])

# idxs = torch.nonzero(a >11)
# print('idxs_size', idxs.size())
# print('idxs', idxs)
# # idxs_size torch.Size([3, 2])

# # print(b.gather(1, idxs))
# import torch
# f = torch.tensor([0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])
# # Create a torch tensor with 20 1-letter strings
# s = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't']
# d1=torch.Tensor(['a','b','c'
# a=torch.tensor([1.0,-2,15, .1, .01, .061,.06,.07,.08,.09,.1,.11,.12,.13,.14,.15,.16,.17,.18,.19,.2,.21,.22,.23,.24,.25,.26,.27,.28,.29,.3,.31,.32,.33,.34,.35,.36,.37,.38,.39,.4,.41,.42,.43,.44,.45,.46,.47,.48,.49,.5,.51,.52,.53,.54,.55,.56,.57,.58,.59,.6,.61,.62,.63,.64,.65,.66,.67,.68,.69,.7,.71,.72,.73,.74,.75,.76,.77,.78,.79,.8,.81,.82,.83,.84,.85,.86,.87,.88,.89,.9,.91,.92,.93,.94,.95,.96,.97,.98,.99])
# l=torch.gt(a,torch.tensor([.7]))
# h=torch.lt(a,torch.tensor([.8]))
# l&h
# (l&h).tolist()

# anchor_embeddings[:10]
# positive_embeddings[:10]
# positives[:10]
# anchors[:10]# 

In [ ]:
y=torch.arange(0,3)
print(y)
z=torch.Tensor([True,False,True])
print(z)
x=torch.Tensor([True,False,True])
print(x)
x=x==True
print(x)
print(y[x])
f = torch.tensor([0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55])
# Create a torch tensor with 20 1-letter strings
s = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l']

# print(get_indices(f))

# get_hard_negatives(f,s)

ff=torch.reshape(f,(4,3))
ss= [['a 1', 'b 1', 'c 1'],[ 'd 1', 'e 1', 'f 1'],['g 1', 'h 1', 'i'], ['j', 'k', 'l']]
get_hard_negatives(ff,ss)


tensor([0, 1, 2])
tensor([1., 0., 1.])
tensor([1., 0., 1.])
tensor([ True, False,  True])
tensor([0, 2])


In [ ]:
%%time
# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
top_k = min(100, len(positive_embeddings))

#for just 1 query
anchor=anchor_embeddings[0]
# We use cosine-similarity and torch.topk to find the highest 5 scores
similarity_scores = model.similarity(anchor, positive_embeddings)[0]
scores, indices = torch.topk(similarity_scores, k=top_k)

gt90=0
gt80=0
gt70=0
lt70=0
for anchor in anchor_embeddings:
    # We use cosine-similarity and torch.topk to find the highest 5 scores
    similarity_scores = model.similarity(anchor, positive_embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_k)
    if scores[0]>90: 
        gt90+=1 
    elif scores[0]>80: 
        gt80+=1 
    elif scores[0]>70: 
        gt70+=1 
    else: 
        lt70+=1
print(f"gt90: {gt90}, gt80: {gt80}, gt70: {gt70}, lt70: {lt70}")
    


gt90: 0, gt80: 0, gt70: 0, lt70: 35258
CPU times: user 14.1 s, sys: 10.3 ms, total: 14.1 s
Wall time: 14.2 s


In [ ]:
# #the following runs on GPU
# import torch
# from itertools import compress
# from tqdm.auto import tqdm
# from numba import njit
# import numpy as np

# def get_scores_processed(scores,high=0.65):
#      # Get the entries below high
#     scores_mask=(scores<high).to(scores.device)

#     #set diagonal to False (dont want true positive included in the hard negatives )
#     mask = (torch.eye(scores_mask.shape[0], scores_mask.shape[0])>0).to(scores.device)
#     scores_mask.masked_fill_(mask, False)
   
#     return (scores.cpu().detach().numpy(),scores_mask.cpu().detach().numpy())

# #the following is compiled into c and runs on a cpu
# @njit
# def get_hard_negatives_CPU(scores,scores_mask,positives,low=0.5, high=0.65, topn=20):
#     """
#     Train a sentencetransformer model, get its average similarity score, use range around that average for hard
#     negatives
#     expects scores to be nxn matrix of similarity scores, nparray
#     expects positives to be a list of n strings
#     expects high to be floats denoting the max acceptable similarity score
#     topn: int, number of hard negatives to return from torch.top_k
#     Get pairs of indices with low<= score <= high
#     returns: list of list of positives whose similarity score is between low and high
#     """
    
#       # use scores_mask to select hard negatives
#     hard_negatives=[]
#     hn_found=0
#     poor_hn_found=0
#     def get_res(candidates,true_positive,low,topn):
#         #get all hard negatives >low
#         res=[pos for score,pos in canditates[:topn] if score>=low]

#         #remove all duplicate entries
#         res=list(set(res))

#         #remove all hits that are the same as the true positive
#         res=[val for val in res if val!=true_positive]
#         return res

#     for i,row in enumerate(scores_mask):
#         #get all the positives and their scores
#         # #make sure none of these positives are the same as the true positive
#         # #this happens when you derive multiple queries from the same positive
#         canditates=[(scores[i,j],positives[j]) for j in range(len(row)) if row[j]==True]

#         #sort it by score
#         canditates.sort(key=lambda x: x[0],reverse=True)
#         # print(f'tup={tup}')

#         # #get all hard negatives >low
#         res=[pos for score,pos in canditates[:topn] if score>=low]

#         #remove all duplicate entries
#         res=list(set(res))

#         #remove all hits that are the same as the true positive
#         res=[val for val in res if val!=positives[i]]
                
#         if(len(res)==0):
#             # print(f"no hard negative for row={i}, using top 1 instead with score={str(canditates[0][0])}")
#             res=[pos for score,pos in canditates[:1] if pos!=positives[i]]
#             poor_hn_found+=1
#         else:
#             hn_found+=1
        
#         hard_negatives.append(res)
#     print(f"hn_found={hn_found}, poor_hn_found={poor_hn_found}")
#     return hard_negatives
